# Module 7: Running it for real, verification and field positioning

You run the lab's regression questions against the fixed agent and record the results in MLflow as a baseline. A later change to the model, the prompt or the tools can then be checked against it. You also take away a checklist that maps a symptom a customer describes to the layer to look at first.

In [ ]:
import labnb as lab
%cd ../agent

## Start from module 7

This puts the agent in the state module 7 starts from. Any edits you made are copied to
`agent/.reset-backup/` first, so nothing you wrote is lost.

The reset restores the code fixes through module 5 and clears the gateway setting. Reconnect
to the lab gateway below before running the assembled agent. Your module 6 tool registration
belongs to your project and survives the reset. Confirm that `estimate_delivery` appears in
the tool list. If it is missing, complete module 6's registration step first.

In [ ]:
!python reset.py 7

In [ ]:
import json
from pathlib import Path
gateway = lab.gateway_url()
if not gateway:
    raise RuntimeError("The lab gateway is unavailable. Check LAB_GATEWAY_URL before continuing.")
config = Path("config.toml")
config.write_text(config.read_text().replace('gateway_url = ""', 'gateway_url = ' + json.dumps(gateway), 1))
lab.tool_list()

## 1. Run the regression set

One case per earlier module, run against the agent as it is now, and each judged from the run record
(tool calls, stop reason, error categories) with deterministic checks rather than an LLM judge. The F-21 case also requires its source ID in the answer, and the bad-depot case checks that the answer acknowledges the failed check. These markers do not assess all possible wording or factual claims.

```mermaid
flowchart LR
  subgraph Cases["Cases"]
    direction TB
    C1["m1: the part number"]
    C3["m3: the Denver chain"]
    C4["m4: the Portland depot"]
    C5["m5: F-21 and the K-900"]
    C6["m6: delivery time"]
  end
  Cases --> AG["The assembled agent"]
  AG --> REC["Run records: tool calls, stop reason, categories"]
  REC --> J{"Fixed checks: records and answer markers"}
  J --> REP["Report, and a named baseline run in MLflow"]
```

In [ ]:
lab.regression(name="my-baseline")

During development, a case that cannot apply yet is reported as SKIP and the rest still run.

**This is the module's success signal:** all six regression cases pass, with 0 FAIL and 0 SKIP.

Read the report the run wrote:

In [ ]:
reports = sorted(lab.RUNS.glob("regression-*.md"))
lab.display(lab.Markdown(reports[-1].read_text() if reports else "No report yet."))

## 2. Check the evidence, not the answers

For each case, confirm the corrected behaviour in its trace: tool calls populate, runs end on the
model's own stop, failures carry a category, retrieval grounds the answer, and the out-of-corpus
question is an explicit gap.

In [ ]:
lab.evidence(columns=("question", "turns", "ended by", "tools run"))
lab.links(("Every trace in this experiment", lab.experiment_url()))

If results change, inspect the saved request's endpoint and the current `config.toml`. The file persists across a workbench restart; reset restores a module's prescribed starting configuration. Each notebook explicitly resets before applying its fixes.

## 3. The baseline

`name="my-baseline"` recorded this run in MLflow as an **evaluation run**: one row per case, each
with its trace and the same PASS, FAIL or SKIP and reason printed above. The verdicts come from the regression set's own fixed checks, including source-citation and failed-check markers in the answer; there is no LLM judge. The run also keeps the `passed`, `failed` and
`skipped` counts, and it is linked to the agent version that produced it, for example
`kestrel-m7-start`, or `kestrel-m7-start-modified` once you've edited the agent.

To compare a later run with this baseline, run the set again under another name, then open the
experiment's **Evaluation runs** tab and select both runs. Each case's two results sit side by side,
and the cases that changed stand out:

In [ ]:
lab.links(("The experiment's Evaluation runs tab", lab.experiment_url("evaluation-runs")))

This baseline is evidence that today's build handles the failures it was built for, and no evidence
at all about failures nobody has thought of yet. The checklist at the end of the report maps each
symptom a customer might describe to the layer to look at first; take it with you.

Two failure modes this lab covered only by discussion: a model or prompt change underneath a working
agent, and cost under real concurrency.

## If the result differs

For the assembled lab, the success signal is **6 PASS, 0 FAIL, 0 SKIP**. SKIP is useful while building earlier modules; it is incomplete acceptance here. A command that finishes or exits successfully does not mean its cases passed. Read each row and reason. A response ending on `length` exhausted its output budget, which includes model reasoning; it is an incomplete case, not proof of a parser failure. The current allowance is 4,096 tokens per model call, so its worst-case token use and latency are higher than the earlier 1,024-token setting. Concurrency capacity has not been measured.

If the delivery case is skipped, confirm `gateway_url` and the project registration from module 6. For tool-call or loop failures, inspect `base_url`, the selected solution and the saved requests. For F-21, verify both retrieval and the answer's source citation. If MLflow is unavailable, the terminal report still records every case, but the baseline has not been recorded: restore access and rerun before comparing evaluations. In MLflow, confirm six case rows, their trace links and assessments, and the same counts as the local report. These fixed checks do not establish general answer quality, concurrency capacity or a fresh deployment.